## Imports

In [41]:
import re
import os
import pickle
import ast
import json
from collections import OrderedDict
from tqdm.auto import tqdm
from dotenv import load_dotenv
from minsearch import Index, VectorSearch
import numpy as np
import openai
from sentence_transformers import SentenceTransformer

load_dotenv()

True

## Vanilla LLM Response

In [3]:
openai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
def get_llm_response(client, prompt, chat_model="gpt-4o-mini"):
    chat_messages = [
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=chat_model,
        messages=chat_messages,
    )
    return response.choices[0].message.content

In [9]:
user_prompt = """
    What are the best practices for configuring HPA (Horizontal Pod Autoscaler) in production environments, 
    and what metrics should we consider for optimal scaling?
"""

In [10]:
print(get_llm_response(openai_client, user_prompt))

Configuring the Horizontal Pod Autoscaler (HPA) in production environments is crucial for ensuring that applications can efficiently handle variable loads while optimizing resource usage. Here are some best practices and metrics to consider:

### Best Practices for Configuring HPA

1. **Understand Workload Characteristics**:
   - Analyze the application's behavior under various loads to understand its resource utilization patterns. This understanding will inform your metrics and thresholds.

2. **Set Appropriate Resource Requests and Limits**:
   - Define `requests` and `limits` for CPU and memory in your pod specifications. HPA uses these values to determine how to scale the pods.

3. **Choose the Right Metrics**:
   - Use the most relevant metrics for your application. Common choices include CPU utilization, memory utilization, and custom metrics that reflect your application's state (e.g., request count, response time).

4. **Define Scaling Policies**:
   - Configure scaling policie

## Agent that uses a search tool on our tech knowledge base to answer questions

In [13]:
def load_object(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File {file_path} does not exist")
    
    if not file_path.endswith('.pkl'):
        raise ValueError(f"File {file_path} is not a pickle file")
    
    with open(file_path, 'rb') as f:
        return pickle.load(f)

In [14]:
section_index =  load_object("data/section_chunks_lexical_index.pkl")

In [45]:
def lexical_search(query):
    results = [
        f"Search Result:\n{result['chunk_content']}" for result in section_index.search(query, num_results=5)
    ]
    return "Search Results:\n\n" + "\n\n".join(results)
    # return results

In [69]:
vector_index = load_object('data/section_chunks_vec_index.pkl')
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

In [70]:
def vector_search(query):
    query_emb = embedding_model.encode(query)
    return vector_index.search(query_emb, num_results=5)

In [54]:
search_tool = {
    "type": "function",
    "function": {
        "name": "lexical_search",
        "description": "Search the tech knowledge base",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the tech knowledge base."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}

In [47]:
openai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [106]:
def get_agent_response(client, system_prompt, user_prompt, chat_model="gpt-4o-mini"):
    chat_messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    response = client.chat.completions.create(
        model=chat_model,
        messages=chat_messages,
        tools=[search_tool]
    )
    
    response_message = response.choices[0].message
    chat_messages.append(response_message)
    # print(chat_messages)
    
    if response_message.tool_calls:
        for tool_call in response_message.tool_calls:
            tool_call_id = tool_call.id
            function_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)
            func_obj = globals().get(function_name)
            tool_response = func_obj(**tool_args)
            chat_messages.append({
                "role": "tool",
                "content": str(tool_response),
                "name": function_name,
                "tool_call_id": tool_call_id
            })
        
        final_response = client.chat.completions.create(
            model=chat_model,
            messages=chat_messages
        )
        
        return final_response.choices[0].message.content
        
    return response.choices[0].message.content

In [107]:
system_prompt = """
    You are a helpful assistant for a course.
    Use the search tool to find relevant information from the course materials
    before answering questions.
    If you can find specific information through search, use it to provide accurate
    answers.
    If the search doesn't return relevant results, let the user know and provide
    general guidance.
"""

In [108]:
user_prompt = """
    What are the best practices for configuring HPA (Horizontal Pod Autoscaler) in production environments, 
    and what metrics should we consider for optimal scaling?
"""

In [109]:
print(get_agent_response(openai_client, system_prompt, user_prompt))

It seems that I wasn't able to retrieve specific best practices and metrics for configuring Horizontal Pod Autoscaler (HPA) from the provided course materials. However, I can provide general guidance based on established practices in the use of HPA in production environments.

### Best Practices for Configuring HPA in Production Environments:

1. **Set Resource Requests and Limits**: Ensure that your pods have well-defined CPU and memory requests and limits. HPA scales based on these configurations, so accurate settings are crucial.

2. **Choose Appropriate Metrics**: Use metrics that reflect the performance of your application under load. Common choices include CPU utilization, memory usage, and custom application metrics.

3. **Establish Scaling Limits**: Define minimum and maximum pod counts to prevent issues such as resource exhaustion or excessive scaling.

4. **Consider Multiple Metrics**: Utilize combinations of different metrics (like both CPU and memory usage) for more granula

## Using PydanticAI library's abstractions to perform the above in a far more simpler manner

In [111]:
from typing import List, Any
import pydantic_ai as pda

In [119]:
def lexical_search(query: str) -> List[Any]:
    return section_index.search(query, num_results=5)

In [120]:
agent = pda.Agent(
    name="Tech Assistant",
    instructions=system_prompt,
    tools=[lexical_search],
    model="gpt-4o-mini"
)

In [123]:
result = await agent.run(user_prompt=user_prompt)
print(result.new_messages)

<bound method AgentRunResult.new_messages of AgentRunResult(output="It appears that I couldn't find specific information regarding best practices for configuring Horizontal Pod Autoscaler (HPA) or detailed metrics related to optimal scaling directly from the course materials provided. However, I can offer general insights on this topic.\n\n### Best Practices for Configuring HPA in Production Environments:\n\n1. **Understand Your Workload**:\n   - Analyze the typical load on your application to choose appropriate metrics to scale on.\n   - Different types of workloads may require different configurations.\n\n2. **Use Appropriate Metrics**:\n   - HPA can scale based on different metrics, including CPU utilization, memory usage, and custom metrics. Choose metrics that most accurately reflect your application's performance.\n   \n3. **Set Reasonable Limits**:\n   - Define minimum and maximum instances (replica count) to avoid resource overcommitment and ensure stability during scaling.\n\n